# TP3 - Procesamiento del Lenguaje Natural (NLP)
**Inteligencia Artificial y Neurociencias**

## 3. Modelos preentrenados

En esta parte se evalúa un **encoder preentrenado en español, BETO** (`dccuchile/bert-base-spanish-wwm-cased`), de dos formas:

1. **Sin fine-tuning:** el encoder queda congelado y se usa solo como extractor de *embeddings*. Sobre esos vectores se entrena una Regresión Logística balanceada (el mismo clasificador que en la parte 2), así que la diferencia con la parte 2 es únicamente la representación del texto.
2. **Con fine-tuning:** se ajustan todos los pesos del encoder junto con una capa de clasificación. Se exploran tres decisiones de entrenamiento: **learning rate**, **cantidad de épocas** y **longitud máxima de secuencia**.

**Protocolo de evaluación** (el mismo de las partes 1 y 2):
- Se usan exactamente los conjuntos de `splits_tp3.csv` (split agrupado por propuesta, ≈70/15/15).
- Todas las decisiones se toman con **validación**. El conjunto de **test se usa una única vez**, al final (sección 3.6).
- Métrica principal: **Macro-F1** respecto de `majority_label`. Se reportan además accuracy, F1 por clase, tiempos de entrenamiento e inferencia y uso de VRAM.

> **Para correr en Colab:** activar la GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*) y subir `splits_tp3.csv` (generado por la parte 1), o ejecutar la celda de carga y subirlo cuando lo pida. Todo el notebook tarda aproximadamente 30–45 minutos en una T4.

### 3.0. Configuración

In [ ]:
!pip -q install datasets

In [ ]:
import os
import gc
import time
import random
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

import torch
from torch.utils.data import DataLoader
import transformers
from transformers import (AutoTokenizer, AutoModel, AutoModelForSequenceClassification,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup)

import sklearn
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (f1_score, accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

SEED = 42
CLASES = ["favor", "against", "neutral"]   # mismo orden que en las partes 1 y 2
label2id = {c: i for i, c in enumerate(CLASES)}
id2label = {i: c for c, i in label2id.items()}
MODELO = "dccuchile/bert-base-spanish-wwm-cased"   # BETO (cased)

sns.set_theme(style="whitegrid")
transformers.logging.set_verbosity_error()   # oculta el aviso de "pesos nuevos" de la cabeza de clasificación

def fijar_semilla(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

fijar_semilla(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USA_AMP = DEVICE.type == "cuda"   # precisión mixta (fp16) solo en GPU

def sincronizar():
    """Espera a que termine el trabajo en la GPU (necesario para medir tiempos correctamente)."""
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

# Hardware / versiones (para reportar costos)
print("Python:", platform.python_version(), "| torch:", torch.__version__,
      "| transformers:", transformers.__version__, "| scikit-learn:", sklearn.__version__)
print("CPU:", platform.processor() or platform.machine(), "| núcleos:", os.cpu_count())
if DEVICE.type == "cuda":
    prop = torch.cuda.get_device_properties(0)
    print(f"GPU: {prop.name} | VRAM total: {prop.total_memory / 1e9:.1f} GB")
else:
    print("ATENCIÓN: no hay GPU disponible. El fine-tuning en CPU es muy lento; "
          "activar la GPU en Colab antes de seguir.")

### 3.1. Carga de datos y particiones

Se reconstruye `df_modelado` igual que en las partes 1 y 2 (se descartan los 150 comentarios sin mayoría) y se asigna a cada comentario su conjunto según `splits_tp3.csv`.

In [ ]:
dataset = load_dataset("SINAI/ALIA-es-discriminative-stance-detection")
df = dataset["train"].to_pandas()

cols_anotadores = ["annotation_1", "annotation_2", "annotation_3"]
df_modelado = df[df["majority_label"].isin(CLASES)].copy().reset_index(drop=True)
df_modelado["label"] = df_modelado["majority_label"]
df_modelado["label_id"] = df_modelado["label"].map(label2id)
df_modelado["id"] = df_modelado["id"].astype(str)

# Nivel de acuerdo entre anotadores (se usa en el análisis de errores)
df_modelado["max_acuerdo"] = df_modelado[cols_anotadores].apply(lambda f: f.value_counts().max(), axis=1)
df_modelado["nivel_acuerdo"] = df_modelado["max_acuerdo"].map({3: "Total (3/3)", 2: "Parcial (2/3)"})

RUTA_SPLITS = "splits_tp3.csv"
if not os.path.exists(RUTA_SPLITS):
    try:
        from google.colab import files
        print("No se encontró splits_tp3.csv: subilo (lo genera la parte 1).")
        files.upload()
    except ImportError:
        raise FileNotFoundError("Falta splits_tp3.csv (lo genera la parte 1).")

splits = pd.read_csv(RUTA_SPLITS, dtype={"id": str})
df_modelado = df_modelado.merge(splits, on="id", how="left", validate="one_to_one")
assert df_modelado["split"].notna().all(), "Hay comentarios sin split asignado"

train_df = df_modelado[df_modelado["split"] == "train"].reset_index(drop=True)
val_df = df_modelado[df_modelado["split"] == "val"].reset_index(drop=True)
test_df = df_modelado[df_modelado["split"] == "test"].reset_index(drop=True)
DFS = {"train": train_df, "val": val_df, "test": test_df}

# Controles: todos los comentarios tienen split y no hay propuestas compartidas
assert len(train_df) + len(val_df) + len(test_df) == len(df_modelado) == 2850
print(f"Train: {len(train_df)} | Validación: {len(val_df)} | Test: {len(test_df)}")
for a, b in [(train_df, val_df), (train_df, test_df), (val_df, test_df)]:
    assert not set(a["target"]) & set(b["target"])

y_train = train_df["label"].to_numpy()
y_val = val_df["label"].to_numpy()
y_test = test_df["label"].to_numpy()

display(pd.crosstab(df_modelado["split"], df_modelado["label"]).loc[["train", "val", "test"], CLASES])

### 3.2. Tokenización y entrada del modelo

BETO trae su propio tokenizador (WordPiece, vocabulario de 31.000 subpalabras, distingue mayúsculas). A diferencia de la parte 2, **no se aplica preprocesamiento**: el modelo fue preentrenado sobre texto natural, así que mayúsculas, acentos, stopwords y signos de puntuación se conservan (las negaciones y las mayúsculas sostenidas, como *"NO SE PUEDE"*, pueden llevar información de postura).

**Entrada del modelo.** La postura se define respecto de una propuesta (en la parte 1 vimos que el mismo texto puede ser *favor* en una propuesta y *against* en otra). Un encoder tipo BERT puede recibir **dos segmentos** separados por `[SEP]` y modelar la relación entre ambos con la atención, algo que la bolsa de palabras de la parte 2 no podía hacer. Por eso se usa como entrada el **par (título de la propuesta, comentario)**:

`[CLS] título de la propuesta [SEP] comentario [SEP]`

No se agrega la `description` porque es muy larga (mediana de 105 palabras, máximo 944) y con ella se superarían fácilmente los 512 tokens. En la evaluación sin fine-tuning (3.3), que es barata, se compara además contra usar solo el comentario.

Se mide la longitud en tokens para elegir las longitudes máximas a probar en 3.4.3.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODELO)

largo_com = [len(x) for x in tokenizer(list(df_modelado["comment"]))["input_ids"]]
largo_par = [len(x) for x in tokenizer(list(df_modelado["target"]), list(df_modelado["comment"]))["input_ids"]]
largos = pd.DataFrame({"comentario": largo_com, "propuesta + comentario": largo_par})
display(largos.describe(percentiles=[.5, .75, .9, .95, .99]).round(1))

LONGITUDES = [64, 128, 256]
display(pd.DataFrame({L: (largos > L).mean() * 100 for L in LONGITUDES}).round(1)
        .rename(columns=lambda L: f"% truncado con max_len={L}"))

ax = largos.plot.hist(bins=50, alpha=0.6, figsize=(9, 4))
for L in LONGITUDES:
    ax.axvline(L, ls="--", c="gray")
    ax.text(L, ax.get_ylim()[1] * 0.92, f" {L}", color="gray")
ax.set_xlabel("Cantidad de tokens (BETO)"); ax.set_ylabel("Cantidad de comentarios")
ax.set_title("Longitud en tokens de la entrada"); plt.tight_layout(); plt.show()

print("Ejemplo de entrada tokenizada:")
print(tokenizer.decode(tokenizer(train_df["target"].iloc[0], train_df["comment"].iloc[0])["input_ids"])[:400])

**Observaciones** *(completar con los resultados)*

- ¿Qué percentil de las entradas entra en 64, 128 y 256 tokens? Con una mediana de 46 palabras por comentario, se espera que 128 tokens cubra la gran mayoría de los casos y que 64 trunque una fracción importante.
- El costo de un transformer crece con la longitud de la secuencia (la atención es cuadrática), así que no conviene usar una longitud mayor que la necesaria.

#### Funciones auxiliares

Se definen la tokenización por conjunto (con *padding* dinámico por batch, para no rellenar todo hasta `max_len`), las métricas y el registro de experimentos.

In [ ]:
collator = DataCollatorWithPadding(tokenizer)

def tokenizar(df_split, max_len, entrada="par", con_labels=False):
    """Devuelve una lista de ejemplos tokenizados (el padding lo hace el collator por batch)."""
    if entrada == "par":
        enc = tokenizer(list(df_split["target"]), list(df_split["comment"]),
                        truncation="longest_first", max_length=max_len)
    elif entrada == "comentario":
        enc = tokenizer(list(df_split["comment"]), truncation=True, max_length=max_len)
    else:
        raise ValueError(entrada)
    ejemplos = [{k: enc[k][i] for k in enc.keys()} for i in range(len(df_split))]
    if con_labels:
        for ej, y in zip(ejemplos, df_split["label_id"]):
            ej["labels"] = int(y)
    return ejemplos

def a_device(batch):
    return {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}

def metricas(y_true, y_pred):
    f1s = f1_score(y_true, y_pred, labels=CLASES, average=None, zero_division=0)
    return {"macro_f1": f1_score(y_true, y_pred, labels=CLASES, average="macro", zero_division=0),
            "accuracy": accuracy_score(y_true, y_pred),
            **{f"f1_{c}": f for c, f in zip(CLASES, f1s)}}

COLS_METRICAS = ["macro_f1", "accuracy"] + [f"f1_{c}" for c in CLASES]

def tamaño_modelo_mb(modelo):
    return sum(p.numel() * p.element_size() for p in modelo.parameters()) / 1e6

### 3.3. BETO sin fine-tuning: embeddings congelados + Regresión Logística

El encoder se usa tal como fue preentrenado, sin modificar sus pesos. Para cada comentario se extrae un vector de 768 dimensiones con tres estrategias de *pooling*:

| Pooling | Qué hace |
|---|---|
| `cls` | Vector del token `[CLS]` de la última capa |
| `mean` | Promedio de los vectores de todos los tokens de la última capa (ignorando el padding) |
| `mean_4` | Promedio de los tokens, promediando además las últimas 4 capas |

BETO no fue preentrenado con un objetivo que haga del `[CLS]` un buen resumen de la oración (solo *masked language modeling*), así que es esperable que el promedio de tokens funcione mejor.

Sobre esos vectores se entrena una Regresión Logística con `class_weight="balanced"` (la misma decisión de desbalance que en la parte 2), estandarizando las features y eligiendo `C` en validación. Se comparan las dos entradas (`comentario` y `par`).

Los embeddings se extraen **una sola vez** para todos los conjuntos (incluido test, que solo se usará en 3.6): la extracción no usa etiquetas, así que no filtra información.

In [ ]:
POOLINGS = ["cls", "mean", "mean_4"]
MAX_LEN_EMB = 256   # cubre prácticamente todas las entradas (ver 3.2)

@torch.no_grad()
def extraer_embeddings(modelo, df_split, entrada, max_len=MAX_LEN_EMB, batch_size=64):
    loader = DataLoader(tokenizar(df_split, max_len, entrada), batch_size=batch_size,
                        shuffle=False, collate_fn=collator)
    salidas = {p: [] for p in POOLINGS}
    for batch in loader:
        batch = a_device(batch)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USA_AMP):
            out = modelo(**batch, output_hidden_states=True)
        mask = batch["attention_mask"].unsqueeze(-1).float()
        ultima = out.last_hidden_state.float()
        ult4 = torch.stack(out.hidden_states[-4:]).float().mean(0)
        salidas["cls"].append(ultima[:, 0].cpu())
        salidas["mean"].append(((ultima * mask).sum(1) / mask.sum(1)).cpu())
        salidas["mean_4"].append(((ult4 * mask).sum(1) / mask.sum(1)).cpu())
    return {p: torch.cat(v).numpy() for p, v in salidas.items()}

encoder = AutoModel.from_pretrained(MODELO).to(DEVICE).eval()
TAMAÑO_BETO_MB = tamaño_modelo_mb(encoder)
print(f"Parámetros del encoder: {sum(p.numel() for p in encoder.parameters()) / 1e6:.0f} M "
      f"({TAMAÑO_BETO_MB:.0f} MB en fp32)")

# Calentamiento (la primera llamada en GPU incluye inicializaciones que no son parte del costo real)
extraer_embeddings(encoder, train_df.head(64), "par")

EMB, T_EMB = {}, {}
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
for entrada in ["comentario", "par"]:
    for split, d in DFS.items():
        sincronizar(); t0 = time.perf_counter()
        EMB[(entrada, split)] = extraer_embeddings(encoder, d, entrada)
        sincronizar(); T_EMB[(entrada, split)] = time.perf_counter() - t0
        print(f"{entrada:10s} | {split:5s}: {len(d)} textos en {T_EMB[(entrada, split)]:.1f} s")
VRAM_EMB_GB = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else np.nan
print(f"VRAM pico durante la extracción: {VRAM_EMB_GB:.2f} GB")

In [ ]:
resultados = []   # registro de todos los experimentos sobre validación

def registrar(grupo, experimento, y_true, y_pred, **extra):
    fila = {"grupo": grupo, "experimento": experimento, **metricas(y_true, y_pred), **extra}
    resultados.append(fila)
    return fila

def clf_embeddings(C):
    return make_pipeline(StandardScaler(),
                         LogisticRegression(C=C, max_iter=5000, class_weight="balanced", random_state=SEED))

for entrada in ["comentario", "par"]:
    for pooling in POOLINGS:
        for C in [0.001, 0.01, 0.1, 1.0]:
            clf = clf_embeddings(C)
            t0 = time.perf_counter()
            clf.fit(EMB[(entrada, "train")][pooling], y_train)
            t_fit = time.perf_counter() - t0
            y_pred = clf.predict(EMB[(entrada, "val")][pooling])
            registrar("sin_finetuning", f"{entrada} | {pooling} | C={C}", y_val, y_pred,
                      entrada=entrada, pooling=pooling, C=C, t_fit_logreg_s=t_fit)

sin_ft = pd.DataFrame([r for r in resultados if r["grupo"] == "sin_finetuning"])
display(sin_ft.sort_values("macro_f1", ascending=False)
        .set_index("experimento")[COLS_METRICAS + ["t_fit_logreg_s"]].head(10).round(3))

In [ ]:
# Mejor C para cada combinación de entrada y pooling
mejor_C = sin_ft.loc[sin_ft.groupby(["entrada", "pooling"])["macro_f1"].idxmax()]
piv = mejor_C.pivot(index="pooling", columns="entrada", values="macro_f1").loc[POOLINGS]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(piv, annot=True, fmt=".3f", cmap="Blues", ax=axes[0], cbar=False)
axes[0].set_title("Macro-F1 en validación (mejor C)")

for (entrada, pooling), g in sin_ft.groupby(["entrada", "pooling"]):
    axes[1].plot(g["C"], g["macro_f1"], marker="o", label=f"{entrada} | {pooling}")
axes[1].set_xscale("log"); axes[1].set_xlabel("C (escala log)"); axes[1].set_ylabel("Macro-F1 (validación)")
axes[1].set_title("Sensibilidad a la regularización"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

MEJOR_SIN_FT = max((r for r in resultados if r["grupo"] == "sin_finetuning"), key=lambda r: r["macro_f1"])
print("Mejor configuración sin fine-tuning:", MEJOR_SIN_FT["experimento"],
      f"→ Macro-F1 (val) = {MEJOR_SIN_FT['macro_f1']:.3f}")

In [ ]:
# Se libera el encoder congelado antes de pasar al fine-tuning
del encoder
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

**Observaciones** *(completar con los resultados)*

- **Pooling:** ¿el promedio de tokens (`mean`, `mean_4`) supera al `[CLS]`, como se esperaba para un modelo sin fine-tuning?
- **Entrada:** ¿agregar el título de la propuesta ayuda? Con el encoder congelado, el vector del par mezcla tema y postura, pero no está entrenado para representar la *relación* entre ambos segmentos.
- **Comparación con la parte 2:** este enfoque reemplaza el TF-IDF por embeddings de BETO con el mismo clasificador. Si no supera al mejor modelo clásico, indica que las representaciones genéricas de un encoder sin ajustar no capturan bien la postura (suelen capturar más el tema y el estilo del texto).
- **Regularización:** con 768 features densas y ≈2.000 ejemplos, valores de `C` bajos (más regularización) suelen funcionar mejor.

### 3.4. BETO con fine-tuning

Se agrega una capa lineal de clasificación sobre el `[CLS]` (`AutoModelForSequenceClassification`) y se entrenan **todos** los pesos del modelo. Se implementa un loop de entrenamiento en PyTorch para controlar la evaluación por época, los tiempos y la memoria.

**Decisiones fijas** (valores estándar para BERT-base, no se exploran):
- Optimizador AdamW, `weight_decay = 0.01`, *warmup* lineal del 10% de los pasos seguido de decaimiento lineal, *gradient clipping* en 1,0.
- `batch_size = 16`, precisión mixta (fp16) en GPU.
- **Desbalance:** *cross-entropy* con pesos de clase `balanced` (inversamente proporcionales a la frecuencia en train), equivalente al `class_weight="balanced"` de la parte 2.
- **Entrada:** par (título de la propuesta, comentario), según lo discutido en 3.2.
- **Selección de época (*early stopping*):** después de cada época se evalúa en validación y se conserva el mejor checkpoint según Macro-F1. Por eso en las tablas se reporta el Macro-F1 de la mejor época (`macro_f1`) y también el de la última (`macro_f1_ultima`).

**Decisiones exploradas** (en forma secuencial, como en la parte 2):
1. **Learning rate** (3.4.1): 1e-5, 2e-5, 3e-5 y 5e-5, con 4 épocas y `max_len = 128`.
2. **Épocas** (3.4.2): con el mejor learning rate se entrena durante 8 épocas y se analiza la curva de pérdida y de Macro-F1 por época.
3. **Longitud máxima de secuencia** (3.4.3): 64, 128 y 256 tokens, con el learning rate y las épocas elegidas.

In [ ]:
BATCH_SIZE = 16
WEIGHT_DECAY = 0.01
WARMUP = 0.1

PESOS_CLASE = torch.tensor(
    compute_class_weight("balanced", classes=np.arange(len(CLASES)), y=train_df["label_id"]),
    dtype=torch.float, device=DEVICE)
print("Pesos de clase:", dict(zip(CLASES, PESOS_CLASE.cpu().numpy().round(3))))

@torch.no_grad()
def predecir(modelo, df_split, max_len, batch_size=64):
    modelo.eval()
    loader = DataLoader(tokenizar(df_split, max_len), batch_size=batch_size,
                        shuffle=False, collate_fn=collator)
    preds = []
    for batch in loader:
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USA_AMP):
            logits = modelo(**a_device(batch)).logits
        preds.append(logits.argmax(-1).cpu())
    return np.array([id2label[i] for i in torch.cat(preds).tolist()])

def fine_tune(lr, epocas, max_len, seed=SEED, devolver_modelo=False, verbose=True):
    """Entrena BETO completo. Evalúa en validación al final de cada época y se queda con la mejor.
    Devuelve (resumen, historia por época, modelo con el mejor checkpoint o None)."""
    fijar_semilla(seed)
    modelo = AutoModelForSequenceClassification.from_pretrained(
        MODELO, num_labels=len(CLASES), id2label=id2label, label2id=label2id).to(DEVICE)

    loader = DataLoader(tokenizar(train_df, max_len, con_labels=True), batch_size=BATCH_SIZE,
                        shuffle=True, collate_fn=collator,
                        generator=torch.Generator().manual_seed(seed))
    optim = torch.optim.AdamW(modelo.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    pasos = len(loader) * epocas
    sched = get_linear_schedule_with_warmup(optim, int(WARMUP * pasos), pasos)
    scaler = torch.amp.GradScaler("cuda", enabled=USA_AMP)
    loss_fn = torch.nn.CrossEntropyLoss(weight=PESOS_CLASE)
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    historia, mejor, mejor_estado, t_train = [], {"macro_f1": -1.0}, None, 0.0
    for epoca in range(1, epocas + 1):
        modelo.train()
        perdidas = []
        sincronizar(); t0 = time.perf_counter()
        for batch in loader:
            labels = batch["labels"].to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USA_AMP):
                logits = modelo(**a_device(batch)).logits
            loss = loss_fn(logits.float(), labels)
            optim.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
            scaler.step(optim)
            scaler.update()
            sched.step()
            perdidas.append(loss.item())
        sincronizar(); t_train += time.perf_counter() - t0

        m = metricas(y_val, predecir(modelo, val_df, max_len))
        historia.append({"epoca": epoca, "loss_train": float(np.mean(perdidas)), **m})
        if m["macro_f1"] > mejor["macro_f1"]:
            mejor = {"epoca": epoca, **m}
            if devolver_modelo:
                mejor_estado = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}
        if verbose:
            print(f"  lr={lr:.0e} max_len={max_len} seed={seed} | época {epoca}/{epocas} | "
                  f"loss={historia[-1]['loss_train']:.4f} | Macro-F1 val={m['macro_f1']:.3f}")

    resumen = {"lr": lr, "epocas": epocas, "max_len": max_len, "seed": seed,
               "mejor_epoca": mejor["epoca"], **{k: mejor[k] for k in COLS_METRICAS},
               "macro_f1_ultima": historia[-1]["macro_f1"],
               "t_train_s": t_train, "t_por_epoca_s": t_train / epocas,
               "vram_pico_GB": torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else np.nan}

    if devolver_modelo:
        modelo.load_state_dict(mejor_estado)
        return resumen, historia, modelo
    del modelo, optim
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return resumen, historia, None

corridas = []     # resumen de cada corrida de fine-tuning (validación)
historias = {}    # métricas por época de cada corrida

def correr(grupo, nombre, **kw):
    resumen, historia, _ = fine_tune(**kw)
    corridas.append({"grupo": grupo, "experimento": nombre, **resumen})
    historias[(grupo, nombre)] = pd.DataFrame(historia)
    return resumen

#### 3.4.1. Learning rate

El learning rate es el hiperparámetro más sensible del fine-tuning: muy bajo y el modelo no llega a adaptarse en pocas épocas; muy alto y puede "olvidar" lo aprendido en el preentrenamiento o quedar atascado prediciendo una sola clase. Los valores recomendados en el paper original de BERT van de 2e-5 a 5e-5.

In [ ]:
LRS = [1e-5, 2e-5, 3e-5, 5e-5]
EPOCAS_LR = 4
MAX_LEN_BASE = 128

for lr in LRS:
    correr("learning_rate", f"lr={lr:.0e}", lr=lr, epocas=EPOCAS_LR, max_len=MAX_LEN_BASE)

tab_lr = pd.DataFrame([c for c in corridas if c["grupo"] == "learning_rate"]).set_index("experimento")
display(tab_lr[["mejor_epoca"] + COLS_METRICAS + ["macro_f1_ultima", "t_train_s", "vram_pico_GB"]].round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for lr in LRS:
    h = historias[("learning_rate", f"lr={lr:.0e}")]
    axes[0].plot(h["epoca"], h["loss_train"], marker="o", label=f"lr={lr:.0e}")
    axes[1].plot(h["epoca"], h["macro_f1"], marker="o", label=f"lr={lr:.0e}")
axes[0].set_title("Pérdida de entrenamiento"); axes[0].set_xlabel("Época"); axes[0].set_ylabel("Cross-entropy ponderada")
axes[1].set_title("Macro-F1 en validación"); axes[1].set_xlabel("Época"); axes[1].set_ylabel("Macro-F1")
axes[1].axhline(MEJOR_SIN_FT["macro_f1"], ls="--", c="gray", label="Sin fine-tuning")
for ax in axes:
    ax.set_xticks(range(1, EPOCAS_LR + 1)); ax.legend()
plt.tight_layout(); plt.show()

LR = tab_lr["macro_f1"].idxmax()
LR = float(tab_lr.loc[LR, "lr"])
print(f"Learning rate elegido: {LR:.0e}")

**Observaciones** *(completar con los resultados)*

- ¿Qué learning rate da el mejor Macro-F1? ¿Hay algún valor con el que el modelo no aprende (Macro-F1 cercano al del clasificador trivial, ≈0,26)?
- Comparar la velocidad de convergencia: con learning rates altos la pérdida baja más rápido, pero el Macro-F1 de validación puede empeorar en las últimas épocas (sobreajuste).
- Recordar que con ≈65 comentarios *neutral* en validación, diferencias de 1–2 puntos de Macro-F1 están dentro del ruido (ver también la variabilidad entre semillas en 3.6).

#### 3.4.2. Cantidad de épocas

Con el learning rate elegido se entrena durante 8 épocas y se registra en cada una la pérdida de entrenamiento y el Macro-F1 (y F1 por clase) en validación. Así se ve a partir de qué época el modelo empieza a sobreajustar.

*Nota:* como el learning rate decae linealmente hasta 0 al final del entrenamiento, la curva de un entrenamiento de 8 épocas no es idéntica a entrenar menos épocas; se usa como guía para elegir cuántas épocas entrenar.

In [ ]:
EPOCAS_MAX = 8
res_ep = correr("epocas", f"lr={LR:.0e} | {EPOCAS_MAX} épocas", lr=LR, epocas=EPOCAS_MAX, max_len=MAX_LEN_BASE)
h = historias[("epocas", f"lr={LR:.0e} | {EPOCAS_MAX} épocas")]
display(h.set_index("epoca").round(3))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(h["epoca"], h["loss_train"], marker="o", c="C3")
axes[0].set_title("Pérdida de entrenamiento"); axes[0].set_xlabel("Época"); axes[0].set_ylabel("Cross-entropy ponderada")
for col in ["macro_f1"] + [f"f1_{c}" for c in CLASES]:
    axes[1].plot(h["epoca"], h[col], marker="o", lw=2.5 if col == "macro_f1" else 1, label=col)
axes[1].set_title("Validación por época"); axes[1].set_xlabel("Época"); axes[1].set_ylabel("F1")
axes[1].legend(fontsize=8)
for ax in axes:
    ax.set_xticks(range(1, EPOCAS_MAX + 1))
plt.tight_layout(); plt.show()

EPOCAS = int(res_ep["mejor_epoca"])
print(f"Mejor época: {EPOCAS} → se entrenará con {EPOCAS} épocas "
      f"(conservando igualmente el mejor checkpoint por validación)")

**Observaciones** *(completar con los resultados)*

- ¿En qué época se alcanza el mejor Macro-F1 de validación? Mientras la pérdida de entrenamiento sigue bajando hacia 0, el Macro-F1 de validación suele estancarse o empeorar: el modelo memoriza el conjunto de entrenamiento (≈2.000 ejemplos es poco para 110 M de parámetros).
- ¿Qué clase se beneficia de entrenar más épocas? Es esperable que *neutral* sea la última en aprenderse y la más inestable.

#### 3.4.3. Longitud máxima de secuencia

Se compara el desempeño y el costo (tiempo y VRAM) con distintas longitudes máximas. Con *padding* dinámico, cada batch se rellena solo hasta su entrada más larga, así que el ahorro de una longitud menor viene únicamente de truncar las entradas largas.

In [ ]:
for L in LONGITUDES:
    correr("max_len", f"max_len={L}", lr=LR, epocas=EPOCAS, max_len=L)

tab_len = pd.DataFrame([c for c in corridas if c["grupo"] == "max_len"]).set_index("experimento")
tab_len["%_truncado"] = [(largos["propuesta + comentario"] > L).mean() * 100 for L in tab_len["max_len"]]
display(tab_len[["max_len", "%_truncado", "mejor_epoca"] + COLS_METRICAS +
                ["t_train_s", "t_por_epoca_s", "vram_pico_GB"]].round(3))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.8))
for ax, col, titulo in zip(axes, ["macro_f1", "t_por_epoca_s", "vram_pico_GB"],
                           ["Macro-F1 (validación)", "Tiempo por época (s)", "VRAM pico (GB)"]):
    ax.bar(tab_len["max_len"].astype(str), tab_len[col], color="C0")
    ax.set_title(titulo); ax.set_xlabel("max_len")
    for i, v in enumerate(tab_len[col]):
        ax.text(i, v, f"{v:.3f}" if col == "macro_f1" else f"{v:.1f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout(); plt.show()

MAX_LEN = int(tab_len.loc[tab_len["macro_f1"].idxmax(), "max_len"])
print(f"Longitud máxima elegida: {MAX_LEN}")

**Observaciones** *(completar con los resultados)*

- ¿Truncar a 64 tokens empeora el resultado? Si la postura suele expresarse al principio del comentario, el efecto puede ser chico.
- ¿Pasar de 128 a 256 mejora algo o solo aumenta el costo? Como pocas entradas superan los 128 tokens (ver 3.2), es esperable que la diferencia en Macro-F1 y en tiempo sea pequeña.
- Si las diferencias de Macro-F1 están dentro del ruido, conviene quedarse con la longitud más barata.

### 3.5. Resumen de los experimentos en validación

In [ ]:
CONFIG_FINAL = {"lr": LR, "epocas": EPOCAS, "max_len": MAX_LEN}
print("Configuración final de fine-tuning:", CONFIG_FINAL)

tab_ft = pd.DataFrame(corridas)
display(tab_ft.set_index(["grupo", "experimento"])[["mejor_epoca"] + COLS_METRICAS +
                                                    ["macro_f1_ultima", "t_train_s", "vram_pico_GB"]].round(3))

# Gráfico resumen: mejor sin fine-tuning y todas las corridas de fine-tuning
resumen_val = pd.concat([
    pd.DataFrame([{"experimento": f"Sin FT: {MEJOR_SIN_FT['experimento']}", "grupo": "sin_finetuning",
                   "macro_f1": MEJOR_SIN_FT["macro_f1"]}]),
    tab_ft.assign(experimento=tab_ft["grupo"] + " | " + tab_ft["experimento"])[["experimento", "grupo", "macro_f1"]],
])
fig, ax = plt.subplots(figsize=(9, 0.4 * len(resumen_val) + 1))
sns.barplot(data=resumen_val, y="experimento", x="macro_f1", hue="grupo", dodge=False, ax=ax)
ax.set_xlabel("Macro-F1 (validación, mejor época)"); ax.set_ylabel("")
ax.set_title("Experimentos de la parte 3"); ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

### 3.6. Evaluación final en test

Recién ahora se usa el conjunto de test, una única vez, para:
1. **BETO sin fine-tuning:** la mejor configuración de 3.3 (entrada, pooling y `C`), con la Regresión Logística entrenada en train.
2. **BETO con fine-tuning:** la configuración final (`CONFIG_FINAL`), entrenada en train con el mejor checkpoint elegido por validación.

**Variabilidad entre semillas.** El fine-tuning con pocos datos es sensible a la inicialización de la cabeza de clasificación y al orden de los batches. Para no reportar un resultado afortunado (o desafortunado), la configuración final se entrena con **3 semillas** y se reporta la media y el desvío en validación y en test.

A diferencia de la parte 2, no se reentrena con train + validación, porque la validación se usa para elegir el mejor checkpoint (*early stopping*).

In [ ]:
res_test = []
preds_test = {}

# --- 1. BETO sin fine-tuning ---
ent, pool, C = MEJOR_SIN_FT["entrada"], MEJOR_SIN_FT["pooling"], MEJOR_SIN_FT["C"]
clf_sin_ft = clf_embeddings(C)
t0 = time.perf_counter(); clf_sin_ft.fit(EMB[(ent, "train")][pool], y_train); t_fit = time.perf_counter() - t0
t0 = time.perf_counter(); y_pred = clf_sin_ft.predict(EMB[(ent, "test")][pool]); t_pred = time.perf_counter() - t0
t_inf = T_EMB[(ent, "test")] + t_pred   # extraer embeddings + clasificar
NOMBRE_SIN_FT = f"BETO sin fine-tuning ({ent} | {pool} | C={C}) + LogReg"
preds_test[NOMBRE_SIN_FT] = y_pred
res_test.append({"modelo": NOMBRE_SIN_FT, **metricas(y_test, y_pred), "seed": SEED,
                 "t_train_s": T_EMB[(ent, "train")] + t_fit,   # embeddings de train + ajuste de la LogReg
                 "t_inf_test_s": t_inf, "t_inf_por_1000_s": t_inf / len(y_test) * 1000,
                 "vram_pico_GB": VRAM_EMB_GB, "tamaño_modelo_MB": TAMAÑO_BETO_MB})
print(NOMBRE_SIN_FT, f"→ Macro-F1 test = {res_test[-1]['macro_f1']:.3f}")

In [ ]:
# --- 2. BETO con fine-tuning (3 semillas) ---
SEEDS_FINAL = [42, 43, 44]
finales_val = []

for seed in SEEDS_FINAL:
    resumen, historia, modelo = fine_tune(**CONFIG_FINAL, seed=seed, devolver_modelo=True)
    finales_val.append(resumen)

    # Calentamiento y luego inferencia cronometrada sobre test
    predecir(modelo, test_df.head(64), MAX_LEN)
    sincronizar(); t0 = time.perf_counter()
    y_pred = predecir(modelo, test_df, MAX_LEN)
    sincronizar(); t_inf = time.perf_counter() - t0

    nombre = f"BETO fine-tuning (seed={seed})"
    preds_test[nombre] = y_pred
    res_test.append({"modelo": nombre, **metricas(y_test, y_pred), "seed": seed,
                     "t_train_s": resumen["t_train_s"], "t_inf_test_s": t_inf,
                     "t_inf_por_1000_s": t_inf / len(y_test) * 1000,
                     "vram_pico_GB": resumen["vram_pico_GB"], "tamaño_modelo_MB": tamaño_modelo_mb(modelo)})
    print(nombre, f"→ Macro-F1 val = {resumen['macro_f1']:.3f} (época {resumen['mejor_epoca']}) | "
                  f"Macro-F1 test = {res_test[-1]['macro_f1']:.3f}")
    del modelo
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

finales_val = pd.DataFrame(finales_val)
print(f"\nMacro-F1 en validación (3 semillas): {finales_val['macro_f1'].mean():.3f} ± {finales_val['macro_f1'].std():.3f}")

In [ ]:
test_tabla = pd.DataFrame(res_test).set_index("modelo")
ft = test_tabla[test_tabla.index.str.startswith("BETO fine-tuning")]
fila_media = ft.mean(numeric_only=True)
fila_std = ft.std(numeric_only=True)
test_tabla.loc["BETO fine-tuning (media de 3 semillas)"] = fila_media
test_tabla.loc["BETO fine-tuning (desvío entre semillas)"] = fila_std
test_tabla.loc[["BETO fine-tuning (media de 3 semillas)", "BETO fine-tuning (desvío entre semillas)"], "seed"] = np.nan

display(test_tabla.drop(columns="seed").round(3))
print(f"Macro-F1 test con fine-tuning: {fila_media['macro_f1']:.3f} ± {fila_std['macro_f1']:.3f}")

**Observaciones** *(completar con los resultados)*

- ¿Cuánto mejora el fine-tuning respecto del encoder congelado? ¿Y respecto del mejor modelo clásico de la parte 2?
- ¿El Macro-F1 de test es similar al de validación? Si es bastante menor, puede indicar que la selección de hiperparámetros y de época sobreajustó la validación (que es chica).
- ¿Qué tan grande es el desvío entre semillas comparado con las diferencias entre configuraciones de 3.4? Si es del mismo orden, las diferencias entre learning rates o longitudes no son concluyentes.
- **Costos:** comparar el tiempo de entrenamiento (en GPU) y de inferencia por cada 1.000 comentarios con los de la parte 2 (en CPU), y el tamaño del modelo (≈440 MB contra unos pocos MB del modelo clásico).

### 3.7. Análisis de resultados

#### 3.7.1. Matrices de confusión y F1 por clase

Para el modelo con fine-tuning se muestra la semilla con Macro-F1 de validación más cercano a la mediana (la selección no usa el test).

In [ ]:
seed_rep = int(finales_val.iloc[(finales_val["macro_f1"] - finales_val["macro_f1"].median()).abs().argmin()]["seed"])
NOMBRE_FT = f"BETO fine-tuning (seed={seed_rep})"
print("Semilla representativa:", seed_rep)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (titulo, nombre) in zip(axes, [("Sin fine-tuning", NOMBRE_SIN_FT), ("Con fine-tuning", NOMBRE_FT)]):
    y_pred = preds_test[nombre]
    cm = confusion_matrix(y_test, y_pred, labels=CLASES, normalize="true")
    ConfusionMatrixDisplay(cm, display_labels=CLASES).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
    ax.set_title(f"{titulo} — Macro-F1 = {f1_score(y_test, y_pred, average='macro'):.3f}")
    ax.set_xlabel("Predicción"); ax.set_ylabel("Etiqueta real (majority_label)")
plt.suptitle("Matrices de confusión en test (normalizadas por fila)"); plt.tight_layout(); plt.show()

print("=== BETO con fine-tuning ===")
print(classification_report(y_test, preds_test[NOMBRE_FT], labels=CLASES, digits=3))

cols_f1 = [f"f1_{c}" for c in CLASES] + ["macro_f1"]
x = np.arange(len(cols_f1))
fig, ax = plt.subplots(figsize=(8, 4))
barras_sin = ax.bar(x - 0.2, test_tabla.loc[NOMBRE_SIN_FT, cols_f1].to_numpy(dtype=float), 0.4,
                    label="Sin fine-tuning")
barras_ft = ax.bar(x + 0.2, fila_media[cols_f1].to_numpy(dtype=float), 0.4,
                   yerr=fila_std[cols_f1].to_numpy(dtype=float), capsize=4,
                   label="Fine-tuning (media ± desvío, 3 semillas)")
ax.bar_label(barras_sin, fmt="%.2f", fontsize=8)
ax.bar_label(barras_ft, fmt="%.2f", fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(cols_f1)
ax.set_ylabel("F1 (test)"); ax.set_title("F1 por clase en test"); ax.set_ylim(0, 1); ax.legend()
plt.tight_layout(); plt.show()

#### 3.7.2. Desempeño según el acuerdo entre anotadores

Si el modelo falla sobre todo en los comentarios donde los anotadores tampoco coincidieron, parte del error se explica por la ambigüedad de las etiquetas (techo de desempeño discutido en la parte 1).

In [ ]:
filas = []
for nombre in [NOMBRE_SIN_FT, NOMBRE_FT]:
    for nivel in ["Total (3/3)", "Parcial (2/3)"]:
        m = (test_df["nivel_acuerdo"] == nivel).to_numpy()
        filas.append({"modelo": nombre, "nivel_acuerdo": nivel, "n": m.sum(),
                      "accuracy": accuracy_score(y_test[m], preds_test[nombre][m]),
                      "macro_f1": f1_score(y_test[m], preds_test[nombre][m], labels=CLASES,
                                           average="macro", zero_division=0)})
display(pd.DataFrame(filas).set_index(["modelo", "nivel_acuerdo"]).round(3))

#### 3.7.3. Ejemplos de errores del modelo con fine-tuning

In [ ]:
errores = test_df.assign(pred=preds_test[NOMBRE_FT])[lambda d: d["pred"] != d["label"]]
print(f"Errores en test: {len(errores)} de {len(test_df)}\n")
for (real, pred), g in errores.groupby(["label", "pred"]):
    if len(g) < 5:
        continue
    print(f"=== Real: {real} | Predicho: {pred} ({len(g)} casos) ===")
    for _, fila in g.sample(min(2, len(g)), random_state=SEED).iterrows():
        print(f"• [{fila['target'][:60]}] {fila['comment'][:250]}")
        print(f"  Anotaciones: {fila[cols_anotadores].tolist()}\n")

#### 3.7.4. Comparación con la parte 2

Si está disponible `resultados_parte2_test.csv` (lo genera la parte 2), se comparan desempeño y costo de los tres enfoques evaluados hasta ahora. La comparación completa entre enfoques (incluidos los LLMs) se hace en la parte 5.

In [ ]:
RUTA_P2 = "resultados_parte2_test.csv"
if os.path.exists(RUTA_P2):
    p2 = pd.read_csv(RUTA_P2, index_col=0)
    p2["t_inf_por_1000_s"] = p2["t_inf_por_1000_ms"] / 1000
    comp = pd.concat([
        p2[["macro_f1", "accuracy", "t_train_s", "t_inf_por_1000_s", "tamaño_modelo_MB"]].assign(enfoque="Clásico (parte 2)"),
        test_tabla.loc[[NOMBRE_SIN_FT, "BETO fine-tuning (media de 3 semillas)"],
                       ["macro_f1", "accuracy", "t_train_s", "t_inf_por_1000_s", "tamaño_modelo_MB"]]
                  .assign(enfoque="Encoder (parte 3)"),
    ])
    display(comp.round(4))

    fig, ax = plt.subplots(figsize=(8, 4.5))
    for i, (nombre, fila) in enumerate(comp.iterrows()):
        ax.scatter(fila["t_inf_por_1000_s"], fila["macro_f1"], s=80, color=f"C{i}", label=nombre[:60])
    ax.set_xscale("log"); ax.set_xlabel("Tiempo de inferencia por 1.000 comentarios (s, escala log)")
    ax.set_ylabel("Macro-F1 (test)"); ax.set_title("Desempeño vs. costo de inferencia")
    ax.legend(fontsize=7, loc="lower right"); plt.tight_layout(); plt.show()
    print("Nota: los tiempos de la parte 2 se midieron en CPU y los de la parte 3 en GPU.")
else:
    print(f"No se encontró {RUTA_P2}: subirlo para ver la comparación (opcional).")

### 3.8. Guardado de resultados

Se guardan los resultados para la comparación entre enfoques (parte 5):
- `resultados_parte3_val.csv`: experimentos sin fine-tuning y corridas de fine-tuning sobre validación.
- `resultados_parte3_epocas.csv`: métricas por época de cada corrida de fine-tuning.
- `resultados_parte3_test.csv`: métricas, tiempos, VRAM y tamaño en test.
- `predicciones_parte3_test.csv`: predicciones en test de cada modelo (para comparaciones por comentario entre enfoques).

In [ ]:
pd.concat([pd.DataFrame([r for r in resultados if r["grupo"] == "sin_finetuning"]),
           pd.DataFrame(corridas),
           finales_val.assign(grupo="final_semillas", experimento=lambda d: "seed=" + d["seed"].astype(str))],
          ignore_index=True).to_csv("resultados_parte3_val.csv", index=False)

pd.concat([h.assign(grupo=g, experimento=e) for (g, e), h in historias.items()],
          ignore_index=True).to_csv("resultados_parte3_epocas.csv", index=False)

test_tabla.assign(**{f"config_{k}": v for k, v in CONFIG_FINAL.items()},
                  gpu=torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "cpu"
                  ).to_csv("resultados_parte3_test.csv")

pd.DataFrame({"id": test_df["id"], "label": y_test,
              "pred_sin_finetuning": preds_test[NOMBRE_SIN_FT],
              **{f"pred_finetuning_seed{s}": preds_test[f"BETO fine-tuning (seed={s})"] for s in SEEDS_FINAL}}
             ).to_csv("predicciones_parte3_test.csv", index=False)
print("Archivos guardados.")

try:   # en Colab, descarga los archivos para no perderlos al cerrar el entorno
    from google.colab import files
    for f in ["resultados_parte3_val.csv", "resultados_parte3_epocas.csv",
              "resultados_parte3_test.csv", "predicciones_parte3_test.csv"]:
        files.download(f)
except ImportError:
    pass

### 3.9. Conclusiones de la parte 3 *(completar con los resultados)*

- **Sin fine-tuning:** los embeddings congelados de BETO + Regresión Logística obtienen un Macro-F1 en test de **X**, **(mejor / peor / similar)** que el modelo clásico de la parte 2 (**…**). El mejor pooling fue **…** y agregar la propuesta **(ayudó / no ayudó)**.
- **Con fine-tuning:** el Macro-F1 en test es **Y ± s** (3 semillas), una mejora de **…** puntos respecto del encoder congelado y de **…** respecto de la parte 2. La mayor parte de la mejora viene de la clase **…**.
- **Decisiones de entrenamiento:** el learning rate **(fue / no fue)** determinante (**…**); el mejor Macro-F1 se alcanzó en la época **…** y después el modelo sobreajusta; la longitud máxima **(importa poco / importa)**: con **…** tokens se obtiene un resultado similar a **…** con **…%** menos de tiempo.
- **Costos:** el fine-tuning requiere GPU (≈**…** GB de VRAM, ≈**…** s de entrenamiento en una T4) y el modelo ocupa ≈440 MB; la inferencia toma ≈**…** s cada 1.000 comentarios en GPU, contra ≈**…** en CPU para el modelo clásico.
- **Limitaciones:** *neutral* sigue siendo la clase más difícil (F1 = **…**), en parte por la ambigüedad de las etiquetas; con validación y test chicos, la varianza entre semillas (**± …**) es comparable a varias de las diferencias entre configuraciones.